## Parameters

In [0]:
dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "07")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")

print(f"Processing Yellow Taxi: {year}-{month}")

## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Build the dynamic Bronze path

In [0]:
bronze_path = (
    f"abfss://bronze@stnyctaxide2026nandi.dfs.core.windows.net/"
    f"yellow_taxi/{year}/{month}"
)

print(f"Bronze path: {bronze_path}")

## Read Bronze Parquet

In [0]:
df_bronze = spark.read.parquet(bronze_path)

print(f"Bronze records: {df_bronze.count()}")

## Validate the Bronze schema

In [0]:
print("Bronze schema:")
df_bronze.printSchema()

## Validate required columns

In [0]:
required_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

missing_columns = [
    col for col in required_columns
    if col not in df_bronze.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Schema validation passed.")
print(f"Required columns found: {len(required_columns)}")

## Rename and cast columns
### Main Silver Satanderdization Step

In [0]:
df_silver = (
    df_bronze
    .select(
        F.col("VendorID").cast("int").alias("vendor_id"),
        F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        F.col("passenger_count").cast("double").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance"),
        F.col("RatecodeID").cast("double").alias("rate_code_id"),
        F.col("store_and_fwd_flag").cast("string").alias("store_and_forward_flag"),
        F.col("PULocationID").cast("int").alias("pickup_location_id"),
        F.col("DOLocationID").cast("int").alias("dropoff_location_id"),
        F.col("payment_type").cast("int").alias("payment_type"),
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("extra").cast("double").alias("extra"),
        F.col("mta_tax").cast("double").alias("mta_tax"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("tolls_amount").cast("double").alias("tolls_amount"),
        F.col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
        F.col("total_amount").cast("double").alias("total_amount"),
        F.col("congestion_surcharge").cast("double").alias("congestion_surcharge"),
        F.col("Airport_fee").cast("double").alias("airport_fee"),
        F.col("cbd_congestion_fee").cast("double").alias("cbd_congestion_fee")
    )
)

## Trim and standardize string columns

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "store_and_forward_flag",
        F.upper(F.trim(F.col("store_and_forward_flag")))
    )
)

print("String standardization completed.")

## Add code mappings

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "payment_type_desc",
        F.when(F.col("payment_type") == 1, "Credit card")
         .when(F.col("payment_type") == 2, "Cash")
         .when(F.col("payment_type") == 3, "No charge")
         .when(F.col("payment_type") == 4, "Dispute")
         .when(F.col("payment_type") == 5, "Unknown")
         .when(F.col("payment_type") == 6, "Voided trip")
         .otherwise("Other")
    )
    .withColumn(
        "rate_code_desc",
        F.when(F.col("rate_code_id") == 1, "Standard rate")
         .when(F.col("rate_code_id") == 2, "JFK")
         .when(F.col("rate_code_id") == 3, "Newark")
         .when(F.col("rate_code_id") == 4, "Nassau or Westchester")
         .when(F.col("rate_code_id") == 5, "Negotiated fare")
         .when(F.col("rate_code_id") == 6, "Group ride")
         .otherwise("Other")
    )
)

print("Code mappings added.")

## Add derived columns
### Enriched Silver dataset with reusable analytical fields

In [0]:
df_silver = (
    df_silver
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .withColumn("dropoff_date", F.to_date("dropoff_datetime"))
    .withColumn("pickup_year", F.year("pickup_datetime"))
    .withColumn("pickup_month", F.month("pickup_datetime"))
    .withColumn("pickup_day", F.dayofmonth("pickup_datetime"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("dropoff_datetime")
            - F.unix_timestamp("pickup_datetime")
        ) / 60.0
    )
    .withColumn(
        "trip_speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance") /
            (F.col("trip_duration_minutes") / 60.0)
        ).otherwise(None)
    )
)

print("Derived columns added.")

## Data Quality checks
### Separate valid records from bad records

In [0]:
dq_df = (
    df_silver
    .withColumn(
        "dq_reason",
        F.when(
            F.col("pickup_datetime").isNull(),
            "Missing pickup_datetime"
        )
        .when(
            F.col("dropoff_datetime").isNull(),
            "Missing dropoff_datetime"
        )
        .when(
            F.col("dropoff_datetime") <= F.col("pickup_datetime"),
            "Invalid trip duration"
        )
        .when(
            F.col("trip_distance").isNull(),
            "Missing trip_distance"
        )
        .when(
            F.col("trip_distance") < 0,
            "Negative trip_distance"
        )
        .when(
            F.col("fare_amount").isNull(),
            "Missing fare_amount"
        )
        .when(
            F.col("fare_amount") < 0,
            "Negative fare_amount"
        )
        .when(
            F.col("total_amount").isNull(),
            "Missing total_amount"
        )
        .when(
            F.col("total_amount") < 0,
            "Negative total_amount"
        )
        .when(
            F.col("pickup_location_id").isNull(),
            "Missing pickup_location_id"
        )
        .when(
            F.col("dropoff_location_id").isNull(),
            "Missing dropoff_location_id"
        )
        .otherwise(None)
    )
)

df_valid = dq_df.filter(F.col("dq_reason").isNull())
df_invalid = dq_df.filter(F.col("dq_reason").isNotNull())

print("DQ rules applied.")

## Data Quality Summary

In [0]:
dq_metrics = dq_df.agg(
    F.count("*").alias("total_count"),
    F.sum(
        F.when(F.col("dq_reason").isNull(), 1).otherwise(0)
    ).alias("valid_count"),
    F.sum(
        F.when(F.col("dq_reason").isNotNull(), 1).otherwise(0)
    ).alias("invalid_count")
).collect()[0]

total_count = dq_metrics["total_count"]
valid_count = dq_metrics["valid_count"]
invalid_count = dq_metrics["invalid_count"]

print(f"Total records   : {total_count}")
print(f"Valid records   : {valid_count}")
print(f"Invalid records : {invalid_count}")

if total_count > 0:
    print(f"Invalid %       : {(invalid_count / total_count) * 100:.2f}%")

In [0]:
dq_summary = (
    dq_df
    .filter(F.col("dq_reason").isNotNull())
    .groupBy("dq_reason")
    .count()
    .orderBy(F.desc("count"))
)

display(dq_summary)

In [0]:
display(
    dq_df
    .filter(F.col("dq_reason").isNotNull())
    .groupBy("dq_reason")
    .agg(
        F.count("*").alias("count"),
        F.min("passenger_count").alias("min_passenger_count"),
        F.max("passenger_count").alias("max_passenger_count"),
        F.min("fare_amount").alias("min_fare_amount"),
        F.max("fare_amount").alias("max_fare_amount"),
        F.min("total_amount").alias("min_total_amount"),
        F.max("total_amount").alias("max_total_amount")
    )
    .orderBy(F.desc("count"))
)

## Add processing metadata - Add Lineage

In [0]:
df_valid = (
    df_valid
    .withColumn(
        "source_file",
        F.lit(f"yellow_tripdata_{year}-{month}.parquet")
    )
    .withColumn("source_year", F.lit(int(year)))
    .withColumn("source_month", F.lit(int(month)))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

df_invalid = (
    df_invalid
    .withColumn(
        "source_file",
        F.lit(f"yellow_tripdata_{year}-{month}.parquet")
    )
    .withColumn("source_year", F.lit(int(year)))
    .withColumn("source_month", F.lit(int(month)))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

print("Lineage metadata added.")

## Duplicate Handling

In [0]:
df_valid = df_valid.dropDuplicates()

print("Exact duplicate records removed from valid dataset.")

## Define Silver and Quarantine paths
### keep valid Silver data and invalid/quarantine data separate

In [0]:
silver_path = (
    f"abfss://silver@stnyctaxide2026nandi.dfs.core.windows.net/"
    f"yellow_taxi/{year}/{month}"
)

quarantine_path = (
    f"abfss://quarantine@stnyctaxide2026nandi.dfs.core.windows.net/"
    f"yellow_taxi/{year}/{month}"
)

print(f"Silver path: {silver_path}")
print(f"Quarantine path: {quarantine_path}")

## Write silver valid data

In [0]:
(
    df_valid
    .write
    .mode("overwrite")
    .parquet(silver_path)
)

print(f"Silver data written successfully: {year}-{month}")

## Write Quarantine Data

In [0]:
(
    df_invalid
    .write
    .mode("overwrite")
    .parquet(quarantine_path)
)

print(f"Quarantine data written successfully: {year}-{month}")

## Final Silver Validation

In [0]:
print("========== SILVER PIPELINE VALIDATION ==========")

print(f"Processing period : {year}-{month}")
print(f"Bronze path       : {bronze_path}")
print(f"Silver path       : {silver_path}")
print(f"Quarantine path   : {quarantine_path}")

print("\nRecord counts:")
print(f"Total records     : {total_count}")
print(f"Valid records     : {valid_count}")
print(f"Invalid records   : {invalid_count}")

if total_count > 0:
    print(f"Valid %           : {(valid_count / total_count) * 100:.2f}%")
    print(f"Invalid %         : {(invalid_count / total_count) * 100:.2f}%")

print("\nValidation completed.")